## Column Transformer :

When we have to apply different encoding on differnet columns.

In [16]:
import numpy as np
import pandas as pd

In [17]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [18]:
df=pd.read_csv("dataset/covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [19]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [20]:
# gender , city : nominal categorical feature : OneHotEncodeing
# cough : ordinal categorical feature : Oridnal Encoding

# There are also some missig values
# age : (maybe scaling)
# fever : simple imputer
# cough : ordinal encoder
# city : one_hot encoder
# has_covid : label encoder

In [21]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [22]:
from sklearn.model_selection import train_test_split

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(["has_covid"], axis=1),
    df["has_covid"],
    test_size=0.2,
    random_state=44
)

In [24]:
X_train

,age,gender,fever,cough,city
79,48,Female,103.0,Mild,Kolkata
38,49,Female,101.0,Mild,Delhi
5,84,Female,NaN,Mild,Bangalore
69,73,Female,103.0,Mild,Delhi
46,19,Female,101.0,Mild,Mumbai
...,...,...,...,...,...
3,31,Female,98.0,Mild,Kolkata
59,6,Female,104.0,Mild,Kolkata
45,72,Male,99.0,Mild,Bangalore
35,82,Female,102.0,Strong,Bangalore


## Without Column Transformer

In [25]:
# adding simple inputer to fever col (filling missing values)
si=SimpleImputer()
X_train_fever=si.fit_transform(X_train[["fever"]])

# also the test data
X_test_fever=si.fit_transform(X_test[["fever"]])

X_train_fever

array([[103.        ],
       [101.        ],
       [100.79166667],
       [103.        ],
       [101.        ],
       [ 99.        ],
       [101.        ],
       [102.        ],
       [100.79166667],
       [100.        ],
       [104.        ],
       [101.        ],
       [ 98.        ],
       [103.        ],
       [103.        ],
       [ 99.        ],
       [100.79166667],
       [ 98.        ],
       [101.        ],
       [ 98.        ],
       [ 98.        ],
       [102.        ],
       [104.        ],
       [103.        ],
       [104.        ],
       [ 98.        ],
       [102.        ],
       [ 99.        ],
       [ 98.        ],
       [104.        ],
       [102.        ],
       [101.        ],
       [100.        ],
       [100.79166667],
       [102.        ],
       [ 99.        ],
       [100.79166667],
       [100.        ],
       [104.        ],
       [100.79166667],
       [101.        ],
       [104.        ],
       [101.        ],
       [ 98

In [26]:
## ordinal encoding -->cough

oe=OrdinalEncoder(categories=[["Mild","Strong"]]) # (min,max) order
X_train_cough=oe.fit_transform(X_train[["cough"]])


## also the test data
X_test_cough=oe.fit_transform(X_test[["cough"]])

X_test_cough

array([[0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [1.],
       [1.],
       [0.],
       [0.],
       [1.]])

In [27]:
## OneHotEncoding : gender,city

ohe=OneHotEncoder(drop="first",sparse_output=False)
X_train_gender_city=ohe.fit_transform(X_train[["gender","city"]])

# also the test data
X_test_gender_city=ohe.fit_transform(X_test[["gender","city"]])

X_train_gender_city.shape

(80, 4)

In [28]:
# Extracting Age
X_train_age=X_train.drop(columns=["gender","fever","cough","city"]).values

# also the test data
X_test_age=X_test.drop(columns=["gender","fever","cough","city"]).values

X_train.shape

(80, 5)

In [29]:
X_train_transformed=np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)

# also the test data
X_test_transformed=np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_test_transformed.shape

(20, 7)

## Column Transformer

In [32]:
from sklearn.compose import ColumnTransformer

transformer=ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),["fever"]), # name of transformer obj , transformer name , columns to apply on
    ('tnf2',OrdinalEncoder(categories=[["Mild","Strong"]]),["cough"]),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),["gender","city"])  
],remainder="passthrough") # tranasformer we want to apply , remainder ( what to do with remaining column)

In [33]:
transformer.fit_transform(X_train).shape

(80, 7)

In [34]:
transformer.fit_transform(X_train).shape

(80, 7)